# Supervised fine-tuning of a base checkpoint

Load the published base stage, fine-tune it on the reference conversation
mixture, and publish the `sft` stage.

The C++ mesh training driver owns the fine-tuning loop, the optimizer step, the
metric record, and the summary. It reads a packed `NCSFT001` shard that
`nc.sft.pack` writes, so Python never steps the optimizer
(`docs/training-seam.md` section 6.2). The base load is a weights-only warm
start, so the run does not inherit the pretraining moments
(`docs/checkpoint-format.md` section 20).


## Setup

The setup clones the repository, reads the Kaggle secrets, builds the shared
library for the T4 (`sm_75`), and checks the card count. See
`docs/checkpoint-format.md` section 15.


In [ ]:
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/ming6ao/nanochat.cpp"
REPO = Path("/kaggle/working/nanochat.cpp")
BASE = Path("/kaggle/working/nanochat-cache")

if not (REPO / ".git").is_dir():
    subprocess.run(["git", "clone", REPO_URL, str(REPO)], check=True)
sys.path[:0] = [
    str(REPO / "python"),
    str(REPO / "notebooks"),
    str(REPO / ".agents" / "skills" / "model-fit" / "scripts"),
]


In [ ]:
import session

sess = session.setup(REPO_URL, REPO, arch="sm_75", devices=1, base=BASE)
print("library:", sess.library)
print("secrets:", sorted(sess.secrets))


In [ ]:
import nanochat_cpp as nc

for device in nc.devices():
    print(device, device.name,
          f"{device.total_memory_bytes / 2**30:.1f} GiB")


## Pull the base stage


In [ ]:
import os

import hub_checkpoint

RUN_NAME = "base-L16-H1024-n8-kv4-s4096"
BASE_REPO_ID = hub_checkpoint.repo_id_for(RUN_NAME)
BASE_REVISION = "base-500"

base_folder = hub_checkpoint.pull_stage(
    BASE_REPO_ID, "base", BASE / "hub", revision=BASE_REVISION,
    token=os.environ.get("HF_TOKEN"))
print("base stage:", base_folder)


## Build the driver config

The driver uses one card, `world_size=1`, `per_rank_batch=1`, `grad_accum=1`,
and `seq=SFT_SEQ`. The optimizer and the schedule come from `args.TrainArgs`,
so the driver cannot drift from the reference fine-tuning rates. The warm start
is the base checkpoint parameters only, and the data is the packed shard the
next cells write.


In [ ]:
import args

model_args = args.ModelArgs.from_json(
    (base_folder / "model.json").read_text(encoding="utf-8"))
config = model_args.to_config()

SFT_SEQ = 128
SFT_STEPS = 40
SFT_TRAIN = args.TrainArgs(
    num_iterations=SFT_STEPS, embedding_lr=0.3, unembedding_lr=0.008,
    matrix_lr=0.02, scalar_lr=0.5, weight_decay=0.0, warmup_steps=2,
    warmdown_ratio=0.5, final_lr_frac=0.0)
optimizer_config = nc.optimizer_config(**SFT_TRAIN.optimizer_kwargs())

SFT_RUN_NAME = f"sft-L{model_args.num_layers}-H{model_args.hidden_dim}"
run_dir = BASE / "runs" / SFT_RUN_NAME
run_dir.mkdir(parents=True, exist_ok=True)
shard_path = run_dir / "sft_rows.nchsft"

train_config = nc.MeshTrainConfig(
    config, preset="track3", world_size=1, optimizer=optimizer_config,
    warm_start_path=str(base_folder / "checkpoint.ckpt"),
    packed_rows_path=str(shard_path),
    per_rank_batch=1, grad_accum=1, steps=SFT_STEPS, seq=SFT_SEQ,
    seed=42, out_dir=str(run_dir))
print("run:", SFT_RUN_NAME, "| shard:", shard_path)


## The base completion


In [ ]:
from kaggle_demo import complete

tokenizer = nc.Tokenizer.load(str(base_folder / "tokenizer.nctoken"))
read_model = nc.Model(config, device="cuda", seed=0)
read_model.load(str(base_folder / "checkpoint.ckpt"))

PROMPT = "The capital of France is"
base_completion = complete(read_model, tokenizer, PROMPT)
print("base completion:", base_completion)


## The conversation mixture

`kaggle_demo.build_sft_conversations` returns a real reference slice when the
Hub is reachable and the offline set otherwise. `nc.sft.pack` packs the
conversations into an `NCSFT001` shard, masks the user turns and the padding
with `-1`, and streams the rows to disk.


In [ ]:
from kaggle_demo import build_sft_conversations

SFT_ROWS = 1000
conversations = build_sft_conversations(limit=SFT_ROWS)
shard = nc.sft.pack(shard_path, conversations, tokenizer, SFT_SEQ)
print(f"packed shard from {len(conversations)} conversations -> {shard}")


## Run the driver

The driver reads the `NCSFT001` shard, cycles the rows, and steps the optimizer.
A target of `-1` is ignored through the masked, valid-token mean in
`src/model.cc`, and Python never steps the optimizer
(`docs/training-seam.md` section 6.2).


In [ ]:
import mesh_train

result = train_config.run()
records = mesh_train.read_metrics(train_config.metrics_path())
losses = [float(row["loss"]) for row in records]
window = max(1, min(5, len(losses) // 2))
early = sum(losses[:window]) / window
late = sum(losses[-window:]) / window
print(f"steps {result.steps}  mean loss {result.mean_loss:.4f}")
print(f"early loss {early:.4f}, late loss {late:.4f}")
assert late < early, "the fine-tuning loss did not decrease"


## Save and publish

The driver wrote the checkpoint, the parameters plus the SFT optimizer state, so
the run can resume. The metadata files match the base stage.


In [ ]:
import datetime
import json
import shutil
import subprocess

(run_dir / "model.json").write_text(model_args.to_json() + "\n",
                                    encoding="utf-8")
(run_dir / "train.json").write_text(SFT_TRAIN.to_json() + "\n",
                                    encoding="utf-8")
shutil.copyfile(base_folder / "tokenizer.nctoken",
                run_dir / "tokenizer.nctoken")

git_revision = subprocess.run(
    ["git", "-C", str(REPO), "rev-parse", "--short", "HEAD"],
    capture_output=True, text=True).stdout.strip()
record = hub_checkpoint.build_run_record(
    stage="sft", run_name=SFT_RUN_NAME, git_revision=git_revision,
    build_key=sess.build_key, cuda_arch="sm_75", step=SFT_STEPS,
    parent_repo_id=BASE_REPO_ID, parent_revision=BASE_REVISION,
    created_utc=datetime.datetime.now(datetime.timezone.utc).isoformat())
(run_dir / "run.json").write_text(
    json.dumps(record, indent=1, sort_keys=True) + "\n", encoding="utf-8")

card = f"""# {SFT_RUN_NAME}

Fine-tuned from `{BASE_REPO_ID}` at `{BASE_REVISION}`, {SFT_STEPS} steps.

- container: `{hub_checkpoint.CONTAINER}`
- source revision: `{git_revision}`
"""
(run_dir / "README.md").write_text(card, encoding="utf-8")
print("saved:", run_dir)


In [ ]:
hub_checkpoint.push_stage(BASE_REPO_ID, "sft", run_dir,
                          token=os.environ.get("HF_TOKEN"),
                          tag=f"sft-{SFT_STEPS}")
print("pushed:", BASE_REPO_ID, "stage sft", f"tag sft-{SFT_STEPS}")


## A small chat check


In [ ]:
from kaggle_demo import complete

read_model.load(str(run_dir / "checkpoint.ckpt"))
print("base completion:", base_completion)
print("sft completion: ", complete(read_model, tokenizer, PROMPT))


## Next

Run `eval.ipynb` with `STAGE = "sft"` to measure the fine-tuned stage on the
reference chat tasks.
